# 工作台 HTTP / MySQL 集成验证

使用公开HTTP边界和真实MySQL。覆盖CSV/Excel预览、字段确认、派生血缘、试算、跨连接保存、幂等保存、任务取消、归档/导出、旧记录和错误响应。
只清理本notebook生成的数据集和记录UUID；不会清空表。运行前显式初始化网页表；凭据读取本机配置或FRM_MYSQL_CONFIG。


In [1]:
from pathlib import Path
import sys
import os
import time
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy.stats import norm
ROOT = Path.cwd().parent if Path.cwd().name == 'api' else Path.cwd()
sys.path.insert(0, str(ROOT))
from fastapi.testclient import TestClient
from data import Database, MySQLConfig
from data.workbench_repository import initialize_workbench
from api.main import create_app
database = Database(MySQLConfig.load())
initialize_workbench(database)
dataset_ids = []
record_ids = []

def checked(response, status=200):
    assert response.status_code == status, (response.status_code, response.text)
    return response.json()

def wait_job(client, identity):
    deadline = time.monotonic() + 45
    while time.monotonic() < deadline:
        job = checked(client.get('/api/jobs/' + identity))
        if job['status'] in ('completed', 'failed', 'cancelled'):
            return job
        time.sleep(0.1)
    raise AssertionError('Task timeout')


D:\FRM2\outputs\runtime\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [2]:
try:
    with TestClient(create_app(database)) as client:
        assert checked(client.get('/api/health'))['status'] == 'ok'
        assert len(checked(client.get('/api/catalog'))['chapters']) == 18
        examples = checked(client.get('/api/examples'))['items']
        example = checked(client.post('/api/previews/example', json={'example_id': examples[0]['id']}))
        assert example['row_count'] > 0

        # 上传只产生预览，百分数到小数由明确步骤转换。
        original = checked(client.post('/api/previews/upload', files={'file': ('returns.csv', b'r,date\n-4,2026-01-01\n-2,2026-01-02\n0,2026-01-03\n1,2026-01-04\n3,2026-01-05\n,2026-01-06', 'text/csv')}))
        fields = {'r': {'kind':'arithmetic_return', 'unit':'percent', 'sign':'profit'}, 'date': {'kind':'date', 'unit':'date', 'sign':'none'}}
        prepared = checked(client.post('/api/previews/' + original['id'] + '/prepare', json={
            'metadata': {'fields': fields}, 'options': {'date_column': 'date', 'sort_by':['date'], 'dropna_columns':['r'], 'conversions':[{'column':'r','kind':'percent_to_decimal'}]}}))
        assert prepared['row_count'] == 5
        assert original['row_count'] == 6
        assert prepared['metadata']['fields']['r']['unit'] == 'decimal'
        saved = checked(client.post('/api/previews/' + prepared['id'] + '/save', json={'name':'API integration returns'}))
        dataset_ids.append(saved['id'])
        dataset_id = saved['id']
        assert checked(client.get('/api/datasets/' + dataset_id))['row_count'] == 5
        assert checked(client.get('/api/datasets?q=API%20integration'))['total'] >= 1

        # Excel工作表选择；原始列未被隐式映射为收益率。
        excel = BytesIO()
        with pd.ExcelWriter(excel, engine='openpyxl') as writer:
            pd.DataFrame({'x':[1,2]}).to_excel(writer, sheet_name='first', index=False)
            pd.DataFrame({'x':[3,4,5]}).to_excel(writer, sheet_name='second', index=False)
        uploaded = checked(client.post('/api/previews/upload', files={'file': ('sample.xlsx', excel.getvalue())}, data={'sheet':'second'}))
        assert uploaded['row_count'] == 3 and uploaded['sheets'] == ['first','second']
        assert uploaded['metadata']['fields']['x']['kind'] == 'other'

        # 保存派生数据，来源ID及步骤保持可追溯。
        derived = checked(client.post('/api/datasets/' + dataset_id + '/prepare', json={'options': {'rename': {'r':'return'}}}))
        child = checked(client.post('/api/previews/' + derived['id'] + '/save', json={'name':'API integration derived'}))
        dataset_ids.append(child['id'])
        assert child['metadata']['lineage']['parent_dataset_id'] == dataset_id
        assert 'r' in checked(client.get('/api/datasets/' + dataset_id))['columns']

        baseline = checked(client.get('/api/records'))['total']
        request = {'dataset_id': dataset_id, 'operation':'var', 'parameters': {'model':'normal','target_column':'r','confidence_level':0.95}}
        started = checked(client.post('/api/jobs', json=request))
        job = wait_job(client, started['id'])
        assert job['status'] == 'completed', job
        assert checked(client.get('/api/records'))['total'] == baseline  # 试算不保存
        expected = -np.mean([-.04,-.02,0,.01,.03]) + np.std([-.04,-.02,0,.01,.03], ddof=1) * norm.ppf(.95)
        np.testing.assert_allclose(job['result']['metrics'][0]['value'], expected)
        with ThreadPoolExecutor(max_workers=2) as pool:
            replies = list(pool.map(lambda _: checked(client.post('/api/jobs/' + started['id'] + '/save', json={'name':'API integration result'})), range(2)))
        assert replies[0]['id'] == replies[1]['id']
        record_id = replies[0]['id']
        record_ids.append(record_id)
        assert checked(client.get('/api/records'))['total'] == baseline + 1
        # 新连接读取真实提交，并保持精确的配置和结果。
        with database.session() as repository:
            persisted = repository.load_calculation(record_id)
            assert persisted['parameters']['request'] == request
        record = checked(client.get('/api/records/' + record_id))
        assert record['result'] == job['result']
        config = client.get('/api/records/' + record_id + '/export?format=json')
        assert config.json()['request'] == request
        assert client.get('/api/records/' + record_id + '/export?format=csv').status_code == 200
        assert 'r' in client.get('/api/datasets/' + dataset_id + '/export').text

        # 重命名与归档不损坏数据、引用或冻结结果。
        checked(client.patch('/api/datasets/' + dataset_id, json={'name':'API renamed','archived':True}))
        checked(client.patch('/api/records/' + record_id, json={'name':'API archived result','archived':True}))
        assert checked(client.get('/api/records/' + record_id))['result'] == job['result']
        assert checked(client.get('/api/datasets/' + dataset_id))['archived'] is True

        failed = checked(client.post('/api/jobs', json={**request, 'parameters': {'model':'normal','target_column':'absent'}}))
        assert wait_job(client, failed['id'])['status'] == 'failed'
        checked(client.post('/api/jobs/' + failed['id'] + '/save', json={'name':'cannot save'}), 409)
        slow = checked(client.post('/api/jobs', json={'dataset_id':dataset_id,'operation':'standard_error', 'parameters':{'method':'bootstrap','model':'normal','target_column':'r','n_resamples':10000,'seed':1}}))
        cancelled = checked(client.post('/api/jobs/' + slow['id'] + '/cancel'))
        assert cancelled['status'] == 'cancelled'
        assert wait_job(client, slow['id'])['status'] == 'cancelled'
        checked(client.post('/api/jobs/' + slow['id'] + '/save', json={'name':'cannot save'}), 409)

        # 老版本记录不要求升级原始数据，可读并明确标为legacy。
        with database.session() as repository:
            legacy_id = repository.save_calculation(dataset_id,'old_calculation',{'confidence':.95},0.123,'0'*64)
        record_ids.append(legacy_id)
        legacy = checked(client.get('/api/records/' + legacy_id))
        assert legacy['legacy'] is True and legacy['legacy_result'] == .123
        checked(client.patch('/api/records/' + legacy_id, json={'archived':True}))
        checked(client.get('/api/datasets/missing'), 404)
        checked(client.get('/api/datasets?limit=0'), 422)
        checked(client.post('/api/previews/' + original['id'] + '/save', json={'name':' '}), 422)
        checked(client.post('/api/jobs', json=request, headers={'origin':'https://outside.example'}), 403)
        print('真实HTTP/MySQL主链、数值锚点、幂等保存、血缘、Excel、取消、归档、导出和旧记录验证通过')
finally:
    # 仅删除本次创建的UUID；依赖表先于被引用记录。
    with database.session() as repository:
        with repository.connection.cursor() as cursor:
            for identity in record_ids:
                cursor.execute('DELETE FROM frm_calculation_ui WHERE run_id=%s', (identity,))
                cursor.execute('DELETE FROM frm_calculations WHERE run_id=%s', (identity,))
            for identity in reversed(dataset_ids):
                cursor.execute('DELETE FROM frm_dataset_ui WHERE dataset_id=%s', (identity,))
                cursor.execute('DELETE FROM frm_datasets WHERE dataset_id=%s', (identity,))


真实HTTP/MySQL主链、数值锚点、幂等保存、血缘、Excel、取消、归档、导出和旧记录验证通过
